In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from typing import Tuple, List, Any, Dict
from sklearn.model_selection import train_test_split
import os
import pandas as pd
from gensim.models.keyedvectors import load_word2vec_format
from utils.utils import download_and_unzip_dataset_from_url, tokenize_tweet

from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

from Models.conv_bi_lstm import ConvBiLSTM
import random

In [2]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
# ----- SENTIMENT ANALYSIS DATASET -----
SA = "SA"

data_dir = "../data"

sa_dataset_zip = "sentiment140.zip"

sentiment140_url = "https://www.kaggle.com/api/v1/datasets/download/kazanova/sentiment140"

known_sentiment_csv_file = "training.1600000.processed.noemoticon.csv"

# ----- WORD2VEC PRETRAINED MODEL -----
WORD2VEC = "word2vec"
word2vec_url = "https://drive.google.com/uc?id=1zQRH1zYBHJ_vU_uMkKvvvwQiZwP5N7wW"
word2vec_zip = "word2vec.zip"
known_word2vec_gz_file = "GoogleNews-vectors-negative300"

# ----- RANDOM STATE -----
seed = 1337

# ----- DATASET SPLIT -----
test_size = 0.2

In [4]:
sa_files: list[str] = download_and_unzip_dataset_from_url(sentiment140_url, data_dir, sa_dataset_zip, SA, force_download_and_unzip=False)

word2vec_files: list[str] = download_and_unzip_dataset_from_url(word2vec_url, data_dir, word2vec_zip, WORD2VEC, force_download_and_unzip=False, use_gdown=True, delete_MACOSX=True)

sa_dataset = None
word2vec_model = None

for file in sa_files:
    if known_sentiment_csv_file in file:
        sa_dataset = file

for file in word2vec_files:
    if known_word2vec_gz_file in file:
        word2vec_model = file

        
if word2vec_model is None or sa_dataset is None:
    raise ValueError("Path to dataset or model could not be found.")

Dataset sentiment140.zip already downloaded, skipping...
Dataset sentiment140.zip already unzipped, skipping...
Dataset word2vec.zip already downloaded, skipping...
Dataset word2vec.zip already unzipped, skipping...


In [5]:
columns = ["target", "id", "date", "flag", "user", "text"]

df = pd.read_csv(sa_dataset, encoding="latin-1", names=columns)

In [6]:
df["tokens"] = df["text"].fillna("").apply(tokenize_tweet)

In [7]:
train, test = train_test_split(df, test_size=test_size, random_state=seed)

In [8]:
w2v_model = load_word2vec_format(word2vec_model, binary = True)

In [9]:
embeddings: torch.Tensor = torch.from_numpy(w2v_model.vectors)

In [10]:
class TweetDataset(Dataset):
    """
    A PyTorch Dataset for the TweetDataset dataset.

    Attributes:
        texts (List[List[str]]): List of tweets tokens.
        targets (List[str]): List of target labels.
    """

    def __init__(self,
                 texts: List[List[str]],
                 targets: List[int]
                 ):
        """
        Initializes the TweetDataset with the given file path.

        Args:
            texts (List[List[str]]): List of tweets tokens.
            targets (List[str]): List of target labels.
        """
        self.texts = texts
        self.targets = targets

    def __len__(self) -> int:
        """Returns the length of the dataset."""
        return len(self.texts)

    def __getitem__(self, idx: int) -> Tuple[List[List[str]], List[int]]:
        """
        Returns the embedded tensor and target for the text at the specified index.

        Args:
            idx (int): Index of the item.

        Returns:
            Tuple[List[str], List[int]]: A tuple containing the texts and the target label for idx.
        """

        return self.texts.values[idx], int(self.targets.values[idx])

In [11]:
def word2idx(embedding_model: Any, tweet: List[str]) -> torch.Tensor:
    """
    Converts a tweet to a list of word indices based on an embedding model.

    This function iterates through each word in the tweet and retrieves its corresponding index
    from the embedding model's vocabulary. If a word is not present in the model's vocabulary,
    it is skipped.

    Args:
        embedding_model (Any): The embedding model with a 'key_to_index' attribute, which maps words to their indices.
        tweet (List[str]): A list of words representing the tweet.

    Returns:
        torch.Tensor: A tensor of word indices corresponding to the words in the tweet.
    """
    return torch.tensor([embedding_model.key_to_index[t] for t in tweet if t in embedding_model.key_to_index])

In [12]:
def collate_fn(batch: List[Tuple[List[str], int]]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Prepares and returns a batch for training/testing in a torch model.

    This function sorts the batch by the length of the text sequences in descending order,
    tokenizes the text using a pre-defined word-to-index mapping, pads the sequences to have
    uniform length, and converts labels to tensor.

    Args:
        batch (List[Tuple[List[str], int]]): A list of tuples, where each tuple contains a
                                             list of words (representing a text) and an integer label.

    Returns:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]: A tuple containing three elements:
            - texts_padded (torch.Tensor): A tensor of padded word indices of the text.
            - labels (torch.Tensor): A tensor of labels.
            - lengths (torch.Tensor): A tensor representing the lengths of each text sequence.
    """
    # TODO: Sort the batch by the length of text sequences in descending order
    batch.sort(key= lambda x: len(x[0]), reverse=True)

    # TODO: Unzip texts and labels from the sorted batch
    texts, labels = zip(*batch)

    # TODO: Convert texts to indices using the word2idx function and w2v_model

    texts_indx: List[torch.Tensor] = [word2idx(w2v_model, text) for text in texts]

    # TODO: Calculate the lengths of each element of texts_indx.
    # The minimum length shall be 1, in order to avoid later problems when training the RNN
    lengths: List[torch.Tensor] = [max(1, text.shape[0]) for text in texts_indx]
    # TODO: Pad the text sequences to have uniform length
    texts_padded: torch.Tensor = pad_sequence(texts_indx).long()
    
    # TODO: Convert labels to tensor
    labels: torch.Tensor = torch.tensor(labels)

    return texts_padded.T, labels, torch.tensor(lengths)

In [13]:
def calculate_accuracy(model: torch.nn.Module, dataloader: DataLoader, embedding_layer: torch.nn.Module, threshold: float = 0.5, device: str = 'cpu') -> float:
    """
    Calculate the accuracy of a PyTorch model given a DataLoader.

    The function moves the model to the specified device, sets it to evaluation mode, and computes
    the accuracy by comparing the model's predictions against the true labels. The predictions are
    determined based on a specified threshold.

    Args:
        model (torch.nn.Module): The PyTorch model to evaluate.
        dataloader (DataLoader): The DataLoader containing the dataset to evaluate against.
        threshold (float, optional): Probability threshold to predict a sample as positive. Defaults to 0.5.
        device (str, optional): Device to which the model and data are moved ('cpu' or 'cuda'). Defaults to 'cpu'.

    Returns:
        float: The accuracy of the model on the given dataset.
    """
    # TODO: Calculate accuracy of a model given a dataloader
    model = model.to(device)
    model.eval()
    embedding_layer = embedding_layer.to(device)
    embedding_layer.eval()
    correct = 0.0
    n_samples = 0
    with torch.no_grad():
      for text, label, text_length in dataloader:
        text = text.to(device).clone()
        label = label.to(device)

        embedded_text = embedding_layer(text)
        outputs = model(embedded_text)
        
        probs = torch.softmax(outputs, dim=1).squeeze()
        
        preds = torch.argmax(probs, dim=1) 

        # Using label // 4 maps target 4 to label 1, and target 0 to label 0
        correct += (preds == label//4).sum().item()
        n_samples += label.shape[0]
        
    accuracy = correct / n_samples

    return accuracy

In [14]:
from tqdm import tqdm

In [15]:
def train_torch_model(model: torch.nn.Module, train_dataloader: DataLoader,
                val_dataloader: DataLoader, criterion: nn.Module,
                optimizer: torch.optim.Optimizer, epochs: int,
                print_every: int, patience: int, embedding_layer: torch.nn.Module,
                device: str = 'cpu', scheduler: torch.optim.lr_scheduler.LRScheduler = None) -> Tuple[Dict[int, float],Dict[int, float]]:
    """
    Train and validate the logistic regression model.

    Args:
        model (torch.nn.Module): An instance of the model to be trained.
        train_dataloader (DataLoader): DataLoader for the training dataset.
        val_dataloader (DataLoader): DataLoader for the validation dataset.
        learning_rate (float): The learning rate for the optimizer.
        criterion (nn.Module): Loss function to use for training.
        optimizer (optim.Optimizer): Optimizer to use for training.
        epochs (int): The number of epochs to train the model.
        print_every (int): Frequency of epochs to print training and validation loss.
        patience (int): The number of epochs to wait for improvement on the validation loss before stopping training early.
        device (str): device where to train the model.

    Returns:
        Tuple[Dict[int, float],Dict[int, float]]: Dictionary of accuracies at each `print_every` interval for the training and validation datasets.
    """
    # TODO: Initialize dictionaries to store training and validation accuracies
    train_accuracies: Dict[int, float] = {}
    val_accuracies: Dict[int, float] = {}

    # TODO: Initialize variables for Early Stopping
    best_loss: float = torch.inf
    epochs_no_improve: int = 0

    # TODO: Move the model to the specified device (CPU or GPU)
    model = model.to(device)
    embedding_layer = embedding_layer.to(device)

    # TODO: Implement the training loop over the specified number of epochs
    for epoch in tqdm(range(epochs)):
        # TODO: Set the model to training mode
        model.train()
        total_loss: int = 0.0

        

        # TODO: Implement the loop for training over each batch in the training dataloader
        for features, labels, text_len in train_dataloader:

            # TODO: Move features and labels to the specified device
            
            features = features.to(device)
            labels = labels.to(device)

            # TODO: Clear the gradients
            optimizer.zero_grad()

            

            # TODO: Forward pass (compute the model output)
            embedded_features = embedding_layer(features)
            outputs = model(embedded_features)
            

            # TODO: Compute the loss
            loss = criterion(outputs, labels//4)

            # TODO: Backward pass (compute the gradients)
            loss.backward()

            # TODO: Update model parameters
            optimizer.step()

            # TODO: Accumulate the loss
            total_loss += loss.item()
            
        total_loss /= len(train_dataloader)
        # TODO: Implement the evaluation phase
        model.eval()
        val_loss: int = 0.0
        with torch.no_grad():
            # TODO: Loop over the validation dataloader
            for features, labels, text_len in val_dataloader:

                # TODO: Move features and labels to the specified device
                features = features.to(device)
                labels = labels.to(device)

                
                # TODO: Forward pass (compute the model output)
                embedded_features = embedding_layer(features)
                outputs = model(embedded_features)
               
                # TODO: Compute the loss
                loss = criterion(outputs, labels//4)

                # TODO: Accumulate validation loss
                val_loss += loss.item()

        val_loss /= len(val_dataloader)

        # TODO: Print training and validation results every 'print_every' epochs
        if epoch % print_every == 0 or epoch == epochs - 1:

            # TODO: Calculate training and validation accuracy
            # accuracy_train = calculate_accuracy(model, train_dataloader, 0.5, device)
            accuracy_val = calculate_accuracy(model, val_dataloader, embedding_layer, 0.5, device)

            # TODO: Store accuracies
            # train_accuracies[epoch] = accuracy_train
            val_accuracies[epoch] = accuracy_val

            # TODO: Calculate and print average losses and accuracies
            print(f"EPOCH {epoch} - Train loss: {total_loss} - Val Loss: {val_loss} - Val Acc: {accuracy_val}")

        
        # TODO: Implement Early Stopping

        if val_loss < best_loss:
            best_loss = val_loss
            epochs_no_improve = 0

        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

        if scheduler != None:
            scheduler.step()

    return train_accuracies, val_accuracies

In [13]:
X_train_val, X_test, y_train_val, y_test = train_test_split(df["tokens"], df["target"], test_size=0.2)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)

In [14]:
tr_dataset = TweetDataset(X_train, y_train)
vl_dataset = TweetDataset(X_val, y_val)
ts_dataset = TweetDataset(X_test, y_test)

In [15]:
batch_size = 128

tr_dataloader: DataLoader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2, drop_last=True)
val_dataloader: DataLoader = DataLoader(vl_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2, drop_last=True)
test_dataloader: DataLoader = DataLoader(ts_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2, drop_last=True)

In [16]:
lr = 1e-4
epochs = 30
print_every = 5
patience = 10
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [17]:

embedding_layer = nn.Embedding.from_pretrained(embeddings).to(device)
embedding_layer.eval()


Embedding(3000000, 300)

In [ ]:

conv_bi_lstm = ConvBiLSTM(embeddings.shape[1], 128, 64, 2, dropout=0.2)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(conv_bi_lstm.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, gamma=0.1, step_size=10)

In [20]:
# Build vocabulary
train_accuracies, val_accuracies = train_torch_model(conv_bi_lstm, tr_dataloader,
                                                                val_dataloader, criterion,
                                                             optimizer, epochs, print_every,
                                                             patience=patience, embedding_layer=embedding_layer, device=device, scheduler=scheduler)

  3%|▎         | 1/30 [01:13<35:41, 73.85s/it]

EPOCH 0 - Train loss: 0.47867095085978506 - Val Loss: 0.45021084117889404 - Val Acc: 0.78834765625


 20%|██        | 6/30 [07:02<28:55, 72.30s/it]

EPOCH 5 - Train loss: 0.4007367544285953 - Val Loss: 0.43405607815086844 - Val Acc: 0.800578125


 37%|███▋      | 11/30 [12:39<22:00, 69.48s/it]

EPOCH 10 - Train loss: 0.3626483434904367 - Val Loss: 0.44681385350227354 - Val Acc: 0.80119140625


 43%|████▎     | 13/30 [15:51<20:44, 73.23s/it]


In [21]:
val_accuracies

{0: 0.78834765625, 5: 0.800578125, 10: 0.80119140625}

In [22]:
accuracy_train = calculate_accuracy(conv_bi_lstm, tr_dataloader, embedding_layer, 0.5, device)
accuracy_val =   calculate_accuracy(conv_bi_lstm, val_dataloader, embedding_layer, 0.5, device)
accuracy_test =  calculate_accuracy(conv_bi_lstm, test_dataloader, embedding_layer, 0.5, device)

print(accuracy_train)
print(accuracy_val)
print(accuracy_test)

0.8588388671875
0.80034765625
0.799440625


In [23]:
def save_model(model: torch.nn.Module, name: str) -> None:
    """
    This function saves a model in the 'models' folder as a torch.jit.
    It should create the 'models' if it doesn't already exist.

    Args:
        model: pytorch model.
        name: name of the model (without the extension, e.g. name.pt).
    """

    # create folder if it does not exist
    if not os.path.isdir("models"):
        os.makedirs("models")

    # save scripted model
    model_scripted = torch.jit.script(model.cpu())
    model_scripted.save(f"models/{name}.pt")

    return None

In [ ]:
save_model(conv_bi_lstm, "../models/sa_models/ConvBiLSTM_v2.0.0_12-04-2026")

In [18]:
def load_model(name: str):
    """
    This function is to load a model from the 'models' folder.

    Args:
        name: name of the model to load.

    Returns:
        model in torchscript.
    """

    # define model
    model = torch.jit.load(f"models/{name}.pt")

    return model

In [20]:
conv_bi_lstm = load_model("ConvBiLSTM_v2.0.0_12-04-2026")

In [21]:
def calculate_f1_score(
    model: torch.nn.Module,
    dataloader: DataLoader,
    embedding_layer: torch.nn.Module,
    device: str = "cpu",
) -> float:
    """Calculate binary F1 score for the sentiment model.

    The model predicts class indices {0, 1}. Ground-truth labels in this notebook
    are {0, 4}, so labels are mapped to binary with `label // 4`.
    """
    model = model.to(device)
    model.eval()
    embedding_layer = embedding_layer.to(device)
    embedding_layer.eval()

    tp = 0
    fp = 0
    fn = 0

    with torch.no_grad():
        for text, label, _ in dataloader:
            text = text.to(device).clone()
            label = (label.to(device) // 4).long()

            embedded_text = embedding_layer(text)
            outputs = model(embedded_text)
            preds = torch.argmax(outputs, dim=1).long()

            tp += ((preds == 1) & (label == 1)).sum().item()
            fp += ((preds == 1) & (label == 0)).sum().item()
            fn += ((preds == 0) & (label == 1)).sum().item()

    denominator = (2 * tp + fp + fn)
    return (2 * tp / denominator) if denominator > 0 else 0.0


f1_train = calculate_f1_score(conv_bi_lstm, tr_dataloader, embedding_layer, device)
f1_val = calculate_f1_score(conv_bi_lstm, val_dataloader, embedding_layer, device)
f1_test = calculate_f1_score(conv_bi_lstm, test_dataloader, embedding_layer, device)

print(f"Train F1: {f1_train:.4f}")
print(f"Val F1:   {f1_val:.4f}")
print(f"Test F1:  {f1_test:.4f}")

/opt/conda/lib/python3.13/site-packages/torch/nn/modules/module.py:1784: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)


Train F1: 0.8386
Val F1:   0.8372
Test F1:  0.8378
